In [1]:
import os, time, numpy as np, pandas as pd, torch
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.metrics import f1_score
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# =================================================================
# 1. DATA GENERATION
# =================================================================
enc_dim, proj_dim = 128, 256
np.random.seed(42)
torch.manual_seed(42)

N_train, N_test = 5000, 1000
train_feats  = torch.FloatTensor(np.random.randn(N_train, enc_dim))
test_feats   = torch.FloatTensor(np.random.randn(N_test,  enc_dim))
train_labels = torch.randint(0, 2, (N_train,))
test_labels  = torch.randint(0, 2, (N_test,))

BATCH_SIZE   = 256
train_loader = DataLoader(TensorDataset(train_feats, train_labels), BATCH_SIZE, shuffle=True,  drop_last=True)
test_loader  = DataLoader(TensorDataset(test_feats,  test_labels),  BATCH_SIZE, shuffle=False)

# =================================================================
# 2. AUGMENTATION  (stronger than original)
# =================================================================
def augment(x, noise=0.1, dropout_p=0.1, scale_jitter=0.1):
    """Gaussian noise + feature dropout + random scaling"""
    x = x + noise * torch.randn_like(x)
    mask  = (torch.rand_like(x) > dropout_p).float()
    x = x * mask
    scale = 1.0 + scale_jitter * (2 * torch.rand(x.size(0), 1, device=x.device) - 1)
    return x * scale

# =================================================================
# 3. MODEL DEFINITIONS  (all projectors now use BN)
# =================================================================

class ProjectionMLP(nn.Module):
    """3-layer MLP projector with BN — standard in modern SSL"""
    def __init__(self, in_dim, hidden=512, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, out_dim)
        )
    def forward(self, x): return self.net(x)

class PredictorMLP(nn.Module):
    """2-layer predictor head for BYOL"""
    def __init__(self, dim=256, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, dim)
        )
    def forward(self, x): return self.net(x)

# ── BYOL ──────────────────────────────────────────────────────────
class BYOL(nn.Module):
    def __init__(self, in_dim, out_dim, tau=0.996):
        super().__init__()
        self.tau = tau
        self.online_proj = ProjectionMLP(in_dim, out_dim=out_dim)
        self.online_pred = PredictorMLP(out_dim)
        self.target_proj = ProjectionMLP(in_dim, out_dim=out_dim)
        self._copy_weights()
        for p in self.target_proj.parameters(): p.requires_grad = False

    def _copy_weights(self):
        for q, k in zip(self.online_proj.parameters(), self.target_proj.parameters()):
            k.data.copy_(q.data)

    @torch.no_grad()
    def ema_update(self):
        for q, k in zip(self.online_proj.parameters(), self.target_proj.parameters()):
            k.data = self.tau * k.data + (1 - self.tau) * q.data

    def forward(self, x1, x2=None, eval_mode=False):
        if eval_mode or x2 is None:
            return self.online_proj(x1)
        z1, z2 = self.online_proj(x1), self.online_proj(x2)
        p1, p2 = self.online_pred(z1), self.online_pred(z2)
        with torch.no_grad():
            t1 = self.target_proj(x1).detach()
            t2 = self.target_proj(x2).detach()
        return p1, p2, t1, t2

# ── SimCLR ────────────────────────────────────────────────────────
class SimCLR(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.encoder = ProjectionMLP(in_dim, out_dim=out_dim)
    def forward(self, x, eval_mode=False):
        z1 = self.encoder(x)
        if eval_mode: return z1
        z2 = self.encoder(augment(x))
        return z1, z2

# ── ConSERT ───────────────────────────────────────────────────────
class ConSERT(nn.Module):
    def __init__(self, in_dim, out_dim, dropout_p=0.15):
        super().__init__()
        self.encoder = ProjectionMLP(in_dim, out_dim=out_dim)
        self.dropout_p = dropout_p

    def forward(self, x, eval_mode=False):
        z = self.encoder(x)
        if eval_mode: return z
        # View-1: feature dropout
        mask = (torch.rand_like(z) > self.dropout_p).float()
        z1 = F.normalize(z * mask, dim=1)
        # View-2: cutoff (zero bottom 20% dims) + noise
        cutoff = int(0.20 * z.size(1))
        idx   = z.abs().argsort(dim=1)[:, :cutoff]
        z2    = z.clone()
        z2.scatter_(1, idx, 0)
        z2    = F.normalize(z2 + 0.05 * torch.randn_like(z2), dim=1)
        return z1, z2

# ── ContraBERT ────────────────────────────────────────────────────
class ContraBERT(nn.Module):
    def __init__(self, in_dim, out_dim, m=0.995):
        super().__init__()
        self.m = m
        self.encoder_q = ProjectionMLP(in_dim, out_dim=out_dim)
        self.encoder_k = ProjectionMLP(in_dim, out_dim=out_dim)
        for p_q, p_k in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            p_k.data.copy_(p_q.data); p_k.requires_grad = False

    @torch.no_grad()
    def update_k(self):
        for p_q, p_k in zip(self.encoder_q.parameters(), self.encoder_k.parameters()):
            p_k.data = p_k.data * self.m + p_q.data * (1. - self.m)

    def forward(self, x, eval_mode=False):
        q = self.encoder_q(x)
        if eval_mode: return q
        with torch.no_grad():
            self.update_k()
            k = self.encoder_k(augment(x))
        return q, k

# =================================================================
# 4. LOSS FUNCTIONS
# =================================================================

def byol_loss(p1, p2, t1, t2):
    """Symmetric cosine similarity loss"""
    l1 = 2 - 2 * F.cosine_similarity(p1, t2.detach(), dim=-1).mean()
    l2 = 2 - 2 * F.cosine_similarity(p2, t1.detach(), dim=-1).mean()
    return (l1 + l2) * 0.5

def nt_xent_loss(z1, z2, temperature=0.07):
    """NT-Xent with in-batch negatives (SimCLR / ConSERT / ContraBERT)"""
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    N  = z1.size(0)
    z  = torch.cat([z1, z2], dim=0)          # 2N × D
    sim = torch.mm(z, z.t()) / temperature    # 2N × 2N
    # Mask self-similarities
    mask = torch.eye(2 * N, device=z.device).bool()
    sim  = sim.masked_fill(mask, -1e9)
    # Positives are (i, i+N) and (i+N, i)
    pos  = torch.cat([torch.arange(N, 2*N), torch.arange(0, N)]).to(z.device)
    loss = F.cross_entropy(sim, pos)
    return loss

# =================================================================
# 5. LR SCHEDULER — cosine with linear warmup
# =================================================================

def get_scheduler(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return float(step) / max(1, warmup_steps)
        progress = float(step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# =================================================================
# 6. TRAINING
# =================================================================
EPOCHS   = 30
TEMP     = 0.07
LR       = 3e-4

def train_model(name, model_class):
    torch.cuda.reset_peak_memory_stats() if torch.cuda.is_available() else None
    model = model_class(enc_dim, proj_dim).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

    total_steps  = EPOCHS * len(train_loader)
    warmup_steps = 2 * len(train_loader)   # 2-epoch warmup
    scheduler    = get_scheduler(opt, warmup_steps, total_steps)

    model.train()
    start = time.time()

    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            opt.zero_grad()

            if name == "BYOL":
                x1, x2 = augment(x), augment(x)
                p1, p2, t1, t2 = model(x1, x2)
                loss = byol_loss(p1, p2, t1, t2)
                model.ema_update()
            else:
                z1, z2 = model(x)
                loss   = nt_xent_loss(z1, z2, temperature=TEMP)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            scheduler.step()
            epoch_loss += loss.item()

        if (epoch + 1) % 10 == 0:
            avg = epoch_loss / len(train_loader)
            print(f"  [{name}] Epoch {epoch+1:02d}/{EPOCHS}  loss={avg:.4f}")

    elapsed  = time.time() - start
    peak_mem = torch.cuda.max_memory_allocated() / 1024**2 if torch.cuda.is_available() else 0
    return model, elapsed, peak_mem

# =================================================================
# 7. EVALUATION ENGINE
# =================================================================

def compute_all_metrics(model, name, time_taken, peak_mem):
    model.eval()
    train_z, train_y, test_z, test_y = [], [], [], []

    with torch.no_grad():
        for x, y in train_loader:
            z = F.normalize(model(x.to(device), eval_mode=True), dim=1)
            train_z.append(z); train_y.append(y)

        t0 = time.time()
        for x, y in test_loader:
            z = F.normalize(model(x.to(device), eval_mode=True), dim=1)
            test_z.append(z); test_y.append(y)
        latency_ms = (time.time() - t0) * 1000 / N_test

    train_z = torch.cat(train_z)
    train_y = torch.cat(train_y).to(device)
    test_z  = torch.cat(test_z)
    test_y  = torch.cat(test_y).to(device)

    # ── Alignment ───────────────────────────────────────────────
    x_sample = train_feats[:200].to(device)
    with torch.no_grad():
        za = F.normalize(model(augment(x_sample), eval_mode=True), dim=1)
        zb = F.normalize(model(augment(x_sample), eval_mode=True), dim=1)
        alignment = F.cosine_similarity(za, zb).mean().item()

    # ── Uniformity ──────────────────────────────────────────────
    idx   = torch.randperm(train_z.size(0))[:1000]
    z_sub = train_z[idx]
    pdist = torch.cdist(z_sub, z_sub).pow(2)
    uniformity = torch.log(torch.exp(-2 * pdist).mean()).item()

    # ── Isotropy & SVR ──────────────────────────────────────────
    pca      = PCA(n_components=10).fit(train_z.cpu().numpy())
    isotropy = np.sum(pca.explained_variance_ratio_[:5]) * 100
    _, S, _  = torch.svd(train_z)
    svr      = (S[0] / S.mean()).item()
    batch_var = train_z.var(dim=0).mean().item()

    # ── Downstream MLP Classifier ───────────────────────────────
    # 3-layer classifier trained for 20 epochs
    clf = nn.Sequential(
        nn.Linear(proj_dim, 128), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(128, 64),  nn.ReLU(),
        nn.Linear(64, 2)
    ).to(device)
    opt_clf = torch.optim.Adam(clf.parameters(), lr=1e-3, weight_decay=1e-4)
    clf_sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt_clf, T_max=20)
    clf_loader = DataLoader(TensorDataset(train_z, train_y), 256, shuffle=True)
    for _ in range(20):
        for z_b, y_b in clf_loader:
            opt_clf.zero_grad()
            F.cross_entropy(clf(z_b), y_b).backward()
            opt_clf.step()
        clf_sched.step()

    clf.eval()
    with torch.no_grad():
        preds = clf(test_z).argmax(1)
    acc = (preds == test_y).float().mean().item() * 100
    f1  = f1_score(test_y.cpu(), preds.cpu(), average='macro') * 100

    # ── STS ─────────────────────────────────────────────────────
    n_pairs = 2000
    i1 = torch.randint(0, len(train_z), (n_pairs,))
    i2 = torch.randint(0, len(train_z), (n_pairs,))
    cos_sim  = F.cosine_similarity(train_z[i1], train_z[i2]).cpu().numpy()
    gold_sim = (train_y[i1] == train_y[i2]).cpu().numpy().astype(float)
    zs_sts, _ = spearmanr(cos_sim, gold_sim)

    class STSHead(nn.Module):
        def __init__(self, d):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(d*4, 256), nn.ReLU(), nn.Dropout(0.2),
                                     nn.Linear(256, 64),  nn.ReLU(),
                                     nn.Linear(64, 1))
        def forward(self, u, v):
            return self.net(torch.cat([u, v, (u-v).abs(), u*v], dim=1))

    sts_m   = STSHead(proj_dim).to(device)
    sts_opt = torch.optim.Adam(sts_m.parameters(), lr=1e-3)
    g_t     = torch.tensor(gold_sim, device=device).float().unsqueeze(1)
    for _ in range(10):
        sts_opt.zero_grad()
        F.mse_loss(sts_m(train_z[i1], train_z[i2]), g_t).backward()
        sts_opt.step()

    it1 = torch.randint(0, len(test_z), (n_pairs,))
    it2 = torch.randint(0, len(test_z), (n_pairs,))
    with torch.no_grad():
        p_test = sts_m(test_z[it1], test_z[it2]).squeeze().cpu().numpy()
        g_test = (test_y[it1] == test_y[it2]).cpu().numpy().astype(float)
    sup_sts, _ = spearmanr(p_test, g_test)

    return {
        "Model":    name,
        "ZS-STS":   f"{zs_sts:.3f}",
        "Sup-STS":  f"{sup_sts:.3f}",
        "Acc%":     f"{acc:.1f}",
        "F1%":      f"{f1:.1f}",
        "Align":    f"{alignment:.3f}",
        "Unif":     f"{uniformity:.3f}",
        "Iso(T5)":  f"{isotropy:.1f}%",
        "SVR":      f"{svr:.2f}",
        "B-Var":    f"{batch_var:.4f}",
        "Time(s)":  f"{time_taken:.1f}",
        "Mem(MB)":  f"{peak_mem:.1f}",
        "Inf(ms)":  f"{latency_ms:.3f}",
    }

# =================================================================
# 8. RUNNER
# =================================================================

def run_all():
    results = []
    models_cfg = [
        ("BYOL",        BYOL),
        ("SimCLR",      SimCLR),
        ("ConSERT",     ConSERT),
        ("ContraBERT",  ContraBERT),
    ]

    for name, cls in models_cfg:
        print(f"\n{'='*55}")
        print(f"  🚀  Training  {name}  ({EPOCHS} epochs, T={TEMP})")
        print(f"{'='*55}")
        model, elapsed, peak_mem = train_model(name, cls)
        metrics = compute_all_metrics(model, name, elapsed, peak_mem)
        results.append(metrics)
        print(f"  ✅  Done — Acc={metrics['Acc%']}%  F1={metrics['F1%']}%  "
              f"Align={metrics['Align']}  Unif={metrics['Unif']}")

    df = pd.DataFrame(results).set_index("Model").T

    print("\n\n" + "="*70)
    print("          ⚔️   SSL COMPARISON MATRIX  (4 MODELS — ENHANCED)  ⚔️")
    print("="*70)
    print(df.to_string())
    print("="*70)

    # ── Per-metric winner highlight ──────────────────────────────
    print("\n🏆  Per-metric winners:")
    df_num = pd.DataFrame(results).set_index("Model")
    higher_better = ["ZS-STS","Sup-STS","Acc%","F1%","Align","B-Var"]
    lower_better  = ["Unif","SVR","Time(s)","Inf(ms)"]

    for col in df_num.columns:
        vals = pd.to_numeric(df_num[col].str.replace('%',''), errors='coerce')
        if vals.isna().all(): continue
        if col in higher_better:
            winner = vals.idxmax()
        elif col in lower_better:
            winner = vals.idxmin()
        else:
            continue
        print(f"  {col:12s} → {winner}  ({df_num.loc[winner, col]})")

run_all()

Using device: cuda

  🚀  Training  BYOL  (30 epochs, T=0.07)
  [BYOL] Epoch 10/30  loss=0.6321
  [BYOL] Epoch 20/30  loss=0.5409
  [BYOL] Epoch 30/30  loss=0.5322
  ✅  Done — Acc=49.8%  F1=49.8%  Align=0.881  Unif=-3.138

  🚀  Training  SimCLR  (30 epochs, T=0.07)
  [SimCLR] Epoch 10/30  loss=0.0034
  [SimCLR] Epoch 20/30  loss=0.0031
  [SimCLR] Epoch 30/30  loss=0.0030
  ✅  Done — Acc=50.7%  F1=50.7%  Align=0.827  Unif=-3.888

  🚀  Training  ConSERT  (30 epochs, T=0.07)
  [ConSERT] Epoch 10/30  loss=0.0024
  [ConSERT] Epoch 20/30  loss=0.0023
  [ConSERT] Epoch 30/30  loss=0.0023
  ✅  Done — Acc=49.1%  F1=49.1%  Align=0.808  Unif=-3.893

  🚀  Training  ContraBERT  (30 epochs, T=0.07)
  [ContraBERT] Epoch 10/30  loss=0.0176
  [ContraBERT] Epoch 20/30  loss=0.0072
  [ContraBERT] Epoch 30/30  loss=0.0049
  ✅  Done — Acc=49.1%  F1=49.1%  Align=0.820  Unif=-3.850


          ⚔️   SSL COMPARISON MATRIX  (4 MODELS — ENHANCED)  ⚔️
Model      BYOL  SimCLR ConSERT ContraBERT
ZS-STS    0.001  -0.